In [ ]:
# imports
import polars as pl  # polars is much faster than pandas for large data
import numpy as np

from helpers import get_latest

In [ ]:
# TODO: what stories were made from these? other than the criminal record one with WyoFile
# TODO: get style guide
# TODO: did we use the processed or original data? - looks like original, but can we use processed to aid in duplicate detection? - ask Fish
# TODO: I think we should start open-sourcing our methodology/scripts
# TODO: are we working with Wyoming again? - no
# TODO: do we want a repeat of last year's graphics, or new stuff? longer term data is available, do we want to make use of it? eg, instead of bar graph comparing 2 years, have faceted line graph for each criminality or other breakdown showing digression from stable pattern
# TODO: what kind of comments and feedback did we get on last year's story?
# TODO: which field office made the arrest is the most reliable column - Denver Field Office covers both Wyoming and Colorado
# TODO: Berkeley deportation data project is very responsive - zero in on Colorado with the caveats
# TODO: criminal history - conviction, charge, other (no charge or conviction) - if you want to know most serious charge (MSC) - must merge with detentions - ask Fish for this
# TODO: timeline: within a week, about the number of arrests and most serious convictions

## Source

[Deportation Data Project](https://deportationdata.org/data/ice.html)


# Arrests

This is the main table we are working with to start. From the description: "Records every time ICE arrests someone, whether or not that arrest results in a decision to detain the person; or issues a Notice to Appear (NTA), the document that starts a deportation case. Note that other agencies also issue NTAs, and those would not appear as arrests in this table. We treat "apprehensions," "arrests," and "administrative arrests" as synonyms."


In [ ]:
arrests = pl.read_excel(get_latest("data", "arrests"))
sorted(list(arrests.schema.keys()))

Appears to be a data issue with impossible date of arrest


In [ ]:
arrests = arrests.with_columns(
    arrest_year=pl.col("apprehension_date").dt.year(),
    arrest_month=pl.col("apprehension_date").dt.month(),
    arrest_day=pl.col("apprehension_date").dt.day(),
)

arrests["apprehension_date"].sort().value_counts()

## Cleaning

First filter for blanks in the Apprehension State field. See if Wyoming or Colorado show up in the Facility State, and then if Detention Facility or Apprehension Landmark Site appear to be in Wyoming or Colorado, and update the Apprehension State accordingly.

Note: facility is probably unrelated to apprehension - people have been being shipped to random facilities


In [ ]:
prob_co_arrests = arrests["apprehension_state"].is_null() & (
    arrests["apprehension_site_landmark"].str.contains("DENVER")
)

arrests.filter(prob_co_arrests)["apprehension_site_landmark"].value_counts()

arrests = arrests.with_columns(
    apprehension_state=pl.when(prob_co_arrests)
    .then(pl.lit("COLORADO"))
    .otherwise(pl.col("apprehension_state"))
)

Then take a look at the other states listed in the Apprehension State field and remove those that don't belong. (More on this later.)

TODO: what does this mean


Then search for unique identifiers using pivot table counts and examine those that are more than one. If they occurred in different years, keep both. If they occurred within a day or two get rid of the older one (often there's removal on one of them, keep that one).

If you plan to compare 2025 to 2025 numbers, take out only the data for the dates available in 2025 (Jan. 20 to the most recent month and date). Don't forget to take out the records from Jan.1-19, 2025. (Why?)

Then it's time for analysis. (will update with a new notebook at some point.)


Then remove duplicates, keep latest or removal or same day most info UNLESS different years then keep both

TODO: this used to be based on ID, but isn't it meaningful that someone is arrested more than once?

Changed to use the "duplicate_likely" column provided by Deportation Data Project


In [ ]:
arrests_duplicates = (
    arrests.filter(arrests["duplicate_likely"])
    .sort(["unique_identifier", "apprehension_date_time"])
    .unique(subset=["unique_identifier"], keep="last")
)

arrests = arrests.filter(~arrests["duplicate_likely"]).vstack(arrests_duplicates)

# Detentions

This is the detentions section. The description: "Records each detention from book-in to book-out at a given detention center for a given individual; most individuals have more than one row in the table because they are transferred between detention centers. Individuals may also be detained more than once and therefore have more than one "stay" in detention. We explain further in the relevant field definitions."
This spreadsheet has two tables that must be combined.


In [ ]:
# create a list of prior two tables, then concantenate them to one; check the record totals
detentions = pl.read_excel(get_latest("data", "detention_stays"))
detentions.schema

In [ ]:
# this gets a list of the detention facilities
det_facility = (
    detentions.groupby(["Detention Facility"]).size().reset_index(name="counts")
)
det_facility.write_csv("det_facility.csv")

In [ ]:
# these are the colorado, wyoming facilities thus far
# TODO: where do we get this?

wy_facilities = [
    "CASPER HOLDROOM",
    "CHEYENNE HOLDROOM",
    "LARAMIE COUNTY JAIL",
    "NATRONA COUNTY JAIL",
    "SWEETWATER COUNTY JAIL",
]

co_facilities = [
    "ALAMOSA HOLDROOM",
    "ARAPAHOE COUNTY JAIL",
    "AURORA CITY JAIL",
    "COLO DEPT OF CORRECTIONS",
    "COLO SPRINGS DEN HSI HOLD",
    "CRAIG HOLDROOM",
    "DENVER CONTRACT DETENTION FACILITY",
    "DENVER COUNTY JAIL",
    "DENVER HEALTH MEDICAL CENTER",
    "DENVER HOLD ROOM",
    "DOUGLAS COUNTY DETENTION CENTER",
    "DURANGO HOLDROOM",
    "GLENWOOD SPRINGS HOLDROOM",
    "GRAND JUNCTION HOLDROOM",
    "JACKSON COUNTY SHERIFF",
    "JEFFERSON COUNTY JAIL",
    "MESA COUNTY JAIL",
    "MOFFAT COUNTY JAIL",
    "OTERO COUNTY DETENTION",
    "PUEBLO COUNTY JAIL",
    "PUEBLO HOLDROOM",
    "SUMMIT COUNTY JAIL",
    "TELLER COUNTY JAIL",
    "UCHEALTH UNV HOSP OF COUINTA COUNTY JAIL",
]

In [ ]:
# this uses the list of colo, wyo to extract the info from the larger detention facility list
co_detain = detentions.filter(pl.col("detention_facility_first").is_in(co_facilities))
co_detain.write_csv("co_wyo_detain.csv")
co_detain.head()

In [ ]:
# get a table with the unique identifer and the charge
charge_lookup = detentions[["Unique Identifier", "MSC Charge"]].drop_duplicates(
    subset="Unique Identifier"
)
charge_lookup.head()

In [ ]:
# adds most serious conviction information, if present, to the arrests table
arrests_with_charge = co_arrests.merge(
    charge_lookup, on="Unique Identifier", how="left"
)
arrests_with_charge.head()

In [ ]:
arrests_with_charge.write_csv("arrests_charges.csv")

## Removals

From the description: "Records every deportation that ICE conducts, with a row for each individual deportation. An individual only has more than one row if that individual was deported more than once. Note that expulsions may occur directly at the border, by CBP, without involving ICE."


In [ ]:
removals = pl.read_excel(get_latest("data", "removals"))
removals.head()

In [ ]:
co__wyo_remove = removals[
    (removals["Docket AOR"] == "Denver Area of Responsibility")
    | (removals["Apprehension State"] == "Colorado")
    | (removals["Apprehension State"] == "Wyoming")
]

In [ ]:
co__wyo_remove.head()

In [ ]:
co__wyo_remove.write_csv("co_wyo_remove.csv")

# Detainers

From the description: "Records all requests to state, county, and municipal jails and prisons either for a person to be held on a detainer or for a notification of release date and time. A detainer is a request to a local jail to hold someone for 48 hours beyond when they otherwise would be released so that ICE can make an arrest in the jail while the individual remains detained."


In [ ]:
detainer = pl.read_excel(get_latest("data", "detainer"))
detainer.schema

In [ ]:
felon_lookup = detainer[
    [
        "Unique Identifier",
        "MSC Conviction Date",
        "Detention Facility",
        "Facility State",
        "Prior Felony Yes No",
        "Violent Misdemeanor Yes No",
        "Aggravated Felony Yes No",
    ]
].drop_duplicates(subset="Unique Identifier")
felon_lookup.head()

In [ ]:
# this adds more information about conviction date, facility and more; mostly used the conviction date
arrests_supplement = arrests_with_charge.merge(
    felon_lookup, on="Unique Identifier", how="left"
)
arrests_supplement.head()


In [ ]:
arrests_supplement.write_csv("arrests_plus.csv")

In [ ]:
detainer.groupby(["Facility AOR"]).size().reset_index(name="counts")

In [ ]:
detainer.groupby(["Facility State"]).size().reset_index(name="counts")

In [ ]:
co_wyo_detainer = detainer[
    (detainer["Facility AOR"] == "Denver Area of Responsibility")
    | (detainer["Facility State"] == "Colorado")
    | (detainer["Facility State"] == "Wyoming")
]
co_wyo_detainer.head()

In [ ]:
co_wyo_detainer.write_csv("co_wyo_detainer.csv")

# Encounters

From the description: "Records every time ICE Enforcement and Removal Operations encounters a person, i.e. considers whether to take enforcement action against a person. This need not mean a physical encounter. Most notably, every time ICE processes a match between FBI book-in information (i.e. to a jail or prison) and ICE database information, that match is logged as an ICE encounter. Generally, if an individual appears in the detainers or arrests table, that individual should appear in this table. An individual might appear in the removals or detentions tables without appearing in the encounters data if Customs and Border Protection initially encounters the person. This is both the largest and the sparsest of the tables, and in many cases, encounters lack a unique ID because the individual lacked an A number (A numbers are generally only given to people with immigrant visas or when they are processed for deportation proceedings)."


In [ ]:
encounters = pl.read_excel(get_latest("data", "encounters"))
encounters.schema

In [ ]:
encounters.groupby(["Responsible AOR"]).size().reset_index(name="counts")

In [ ]:
co_encounter = encounters[
    encounters["Responsible AOR"] == "Denver Area of Responsibility"
]
co_encounter.head()

In [ ]:
co_encounter.write_csv("co_encounter.csv")

In [ ]:
# have not looked at this closely
encounters.group_by(["Responsible Site"]).size().reset_index(name="counts")

In [ ]:
# TODO: transfer the excel cleaning steps to python
# TODO: what is "wrong date"?


# Research Questions

- For each year(s), compare...
- How many people were deported to a third country?
- How many people had a criminal record?


In [ ]:
# TODO: follow up with fish on analysis notebook
# gender, age
# citizenship, third country deportations
# per month
# criminality
# TODO: fish comment about which days there weren't any arrests, pulling out holidays
# TODO: Aurora Detention Center operated by Geogroup - for the Aurora specific story - Taylor idea about https://www.nytimes.com/interactive/2025/12/22/us/trump-immigration-deportation-network-ice-arrests.html - inspiration https://coloradotimesrecorder.com/2026/03/after-co-times-recorder-revealed-secret-detention-centers-in-co-27-lawmakers-call-on-ice-for-immediate-transparency/77113/ - Denver Health


# Aurora story
# detentions stays/stints
# TODO: ask DDP if the historical data is comparable for this use case - ask Fish if the historical data changes from release to release


### Number of arrests

In [ ]:
# This gets arrests with Denver area of responsibility and/or the state of Colorado/Wyoming
co_arrests = arrests.filter(
    (arrests["apprehension_aor"] == "Denver Area of Responsibility")
    | (arrests["apprehension_state"] == "Colorado")
    # | (arrests['apprehension_state'] == 'Wyoming')
)

print(co_arrests.shape)
co_arrests.head()

In [ ]:
co_arrests.write_csv("colo_arrests.csv")

For comparing to prior year, ensure that data for prior year up to current month is kept


In [ ]:
by_month_year = (
    co_arrests.filter(co_arrests["arrest_month"].lt(4))
    .group_by(["arrest_day", "arrest_month", "arrest_year"])
    .agg(pl.count())
)


In [ ]:
co_latest = co_arrests.filter(
    (co_arrests["apprehension_date"] <= co_arrests["apprehension_date"].max())
    & (co_arrests["arrest_month"] <= co_arrests["apprehension_date"].max().month)
)

# TODO: remove any dates above 3/1 and also sum 2/29 week with the 3/1 week

by_month_year = co_latest.with_columns(
    arrest_week=(pl.col("arrest_day").cast(pl.Int32) / 7).floor() * 7 + 1
)

by_month_year = by_month_year.with_columns(
    arrest_month_week=pl.col("arrest_month").cast(pl.Utf8).str.zfill(2)
    + "-"
    + pl.col("arrest_week").cast(pl.Int32).cast(pl.Utf8).str.zfill(2)
)

by_month_year = by_month_year.group_by(["arrest_month_week", "arrest_year"]).agg(
    pl.count()
)

by_month_year = by_month_year.pivot(
    index=["arrest_month_week"], columns="arrest_year", values="count"
)

# by_month_year = by_month_year.group_by(["arrest_month", "arrest_year"]).agg(pl.count())

# by_month_year = by_month_year.pivot(
#     index=["arrest_month"], columns="arrest_year", values="count"
# )

by_month_year.write_csv(f"by_month_year_{datetime.today().year}.csv")
by_month_year


In [ ]:
by_day_year = co_latest.group_by(["arrest_day", "arrest_month", "arrest_year"]).agg(
    pl.count()
)

by_day_year = by_day_year.pivot(
    index=["arrest_day", "arrest_month"], columns="arrest_year", values="count"
)

by_day_year.write_csv(f"by_day_year_{datetime.today().year}.csv")
by_day_year


### Criminality

In [ ]:
co_latest.group_by(["apprehension_criminality", "arrest_year"]).agg(pl.count()).pivot(
    values="count", index="arrest_year", columns="apprehension_criminality"
).write_csv(f"by_criminality_{datetime.today().year}.csv")

### MSC (most serious charge) 

percentage 2011-2026 
- merge arrests with detentions on ID to get MSC 
- when they have conviction find MSC 
- note how many people in detention have MSC but no conviction 
- FBI definition of violent/not over time 
- find list of top 5 crimes since Trump's inauguration


In [ ]:
convicted = co_latest.filter(pl.col("apprehension_criminality").str.contains("Convicted"))[['apprehension_criminality', 'arrest_year', 'unique_identifier']]